# 🌿 식물공장 상추 수확량 예측 시스템 v5.1
**Google Drive 연동** · 주수 + kg · 3일 대시보드 · 달별 전체 뷰 · 실적 반영

---

### 📁 Google Drive 폴더 구조
```
내 드라이브/
└── 식물공장_수확예측/
    ├── DB_배치데이터.csv
    └── 예측결과/
        └── prediction_YYYY-MM-DD.csv
```

### 실행 순서
1. **셀 1** — 패키지 설치 (최초 1회)
2. **셀 2** — Google Drive 연결 + DB 초기화
3. **셀 3** — 설정값 (로스율 · MGS · 기준일)
4. **셀 4** — 배치 DB 관리 (추가 / 조회 / 삭제)
5. **셀 5** — 📊 3일 대시보드 (예측 + 실적 비교)
6. **셀 6** — 📅 달별 전체 예측 뷰
7. **셀 7** — 실제 수확 실적 입력
8. **셀 8** — 노션 마크다운 출력
9. **[참고]** — 재배대 전체 용량표

### v4 변경 사항
- 배치 상세 테이블에 **파종일**, **총 재배일수** 컬럼 추가
- **달별 전체 예측 뷰** : DB에 데이터가 있는 모든 월을 수확일 기준으로 집계
- `actual_weight_kg` 입력 시 대시보드에 **예측 vs 실적 비교** 자동 표시
### v5.1 변경 사항
- 배치 상세 컬럼 순서: 파종일 → 수확예정일 → 총 재배일
- 실제 주당무게(g) 컬럼 추가 (비고 바로 앞) — actual_yield·actual_weight_kg 입력 시 자동 계산


In [ ]:
# ============================================================
# 셀 1: 패키지 설치 (최초 1회)
# ============================================================
!pip install pandas numpy -q
print('✅ 패키지 준비 완료')

In [ ]:
# ============================================================
# 셀 2: Google Drive 연결 + DB 초기화
# ============================================================
import os, warnings
import pandas as pd
import numpy as np
from datetime import date, timedelta
from IPython.display import display, HTML
from google.colab import drive
warnings.filterwarnings('ignore')

drive.mount('/content/drive')

BASE_DIR   = '/content/drive/MyDrive/식물공장_수확예측'
RESULT_DIR = os.path.join(BASE_DIR, '예측결과')
DB_PATH    = os.path.join(BASE_DIR, 'DB_배치데이터.csv')
os.makedirs(BASE_DIR,   exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)

DB_COLS = [
    'batch_id',           # 배치 고유 ID
    'sow_date',           # 파종일
    'transplant_date',    # 이식일 (없으면 빈칸)
    'plant_date',         # 정식일
    'harvest_date',       # 수확 예정일
    'grow_days',          # 생육일수 (정식일~수확일)
    'bed_type',           # fixed / mgs
    'bed_id',             # 재배대 번호 또는 MGS 구역명
    'tray_or_gutter',     # 판 수(fixed) 또는 거터 수(mgs)
    'weight_per_plant_g', # 주당 예상 무게(g) — 빈칸이면 DEFAULT_WEIGHT_G 사용
    'loss_rate',          # 로스율 0.0~1.0 — 빈칸이면 LOSS_RATE 사용
    'actual_yield',       # 실제 수확 주수 (수확 후 입력)
    'actual_weight_kg',   # 실제 수확 무게 kg (수확 후 입력)
    'note',               # 비고
]

if not os.path.exists(DB_PATH):
    pd.DataFrame(columns=DB_COLS).to_csv(DB_PATH, index=False, encoding='utf-8-sig')
    print(f'✅ DB 신규 생성: {DB_PATH}')
else:
    _ex = pd.read_csv(DB_PATH, encoding='utf-8-sig')
    for c in DB_COLS:           # 이전 버전 DB에 없는 컬럼 자동 추가
        if c not in _ex.columns:
            _ex[c] = None
    _ex.to_csv(DB_PATH, index=False, encoding='utf-8-sig')
    print(f'✅ DB 연결 완료: {DB_PATH}  ({len(_ex)}건)')

print(f'\n📁 드라이브 경로: {BASE_DIR}')

In [ ]:
# ============================================================
# 셀 3: ⚙️ 설정값 — 매 예측 시 여기만 수정
# ============================================================

PREDICTION_DATE   = date.today()          # 예측 기준일 (오늘 자동)
# PREDICTION_DATE = date(2025, 3, 24)     # 직접 지정 시 주석 해제

LOSS_RATE_PCT     = 20                    # 로스율 (%) — 기본 20% = 수확률 80%
DEFAULT_WEIGHT_G  = 100                   # 주당 기본 무게 (g)
MGS_DEFAULT_WEIGHT_G = 100                # MGS 주당 기본 무게 (g) — 고정 재배대와 별도 설정
# MGS_DEFAULT_WEIGHT_G = 120               # 품종이 다를 때 별도 지정

# ── 고정 설정 (변경 불필요) ──────────────────────────────────
FIXED_BED_CONFIG = {
    1:40,  2:40,  3:40,  4:40,  5:40,  6:40,  7:40,
    8:32,  9:32,  10:32, 11:32, 12:32, 13:32, 14:32,
    15:32, 16:32, 17:32, 18:32,
    19:40, 20:40
}
PLANTS_PER_TRAY   = 16
PLANTS_PER_GUTTER = 13
PREDICTION_WINDOW = [3, 4]

LOSS_RATE  = LOSS_RATE_PCT / 100
YIELD_RATE = 1 - LOSS_RATE
DASH_DATES = [
    PREDICTION_DATE,
    PREDICTION_DATE + timedelta(days=3),
    PREDICTION_DATE + timedelta(days=4),
]
TARGET_DATES = DASH_DATES[1:]
total_cap    = sum(t * PLANTS_PER_TRAY for t in FIXED_BED_CONFIG.values())

print('=' * 54)
print(f'  예측 기준일      : {PREDICTION_DATE}')
print(f'  수확 예정일      : {TARGET_DATES[0]}  /  {TARGET_DATES[1]}')
print(f'  로스율           : {LOSS_RATE_PCT}%  →  수확률 {100-LOSS_RATE_PCT}%')
print(f'  주당 기본 무게   : {DEFAULT_WEIGHT_G} g')
print(f'  MGS 주당 무게    : {MGS_DEFAULT_WEIGHT_G} g  (고정 재배대 기본: {DEFAULT_WEIGHT_G} g)')
print(f'  고정 재배대 용량 : 최대 {total_cap:,}주')
print('=' * 54)

In [ ]:
# ============================================================
# 셀 4: 📋 배치 DB 관리
# ============================================================
# MODE: 'view' | 'add' | 'delete' | 'clear'
# ============================================================

MODE = 'add'

# ── [add] 추가할 배치 목록 ────────────────────────────────────
# weight_per_plant_g : 주당 예상 무게(g). None → DEFAULT_WEIGHT_G 자동 적용
# loss_rate          : None → LOSS_RATE 자동 적용
# tray_or_gutter     : MGS 거터 수 모르면 None
# sow_date           : 파종일 (YYYY-MM-DD). 대시보드 파종일 컬럼에 표시됨
# ─────────────────────────────────────────────────────────────
NEW_BATCHES = [
    {
        'batch_id':           'BATCH-F-01',
        'sow_date':           '2025-03-01',
        'transplant_date':    '2025-03-08',
        'plant_date':         '2025-03-10',
        'harvest_date':       str(TARGET_DATES[0]),
        'grow_days':          17,
        'bed_type':           'fixed',
        'bed_id':             1,
        'tray_or_gutter':     40,
        'weight_per_plant_g': 100,
        'loss_rate':          None,
        'actual_yield':       None,
        'actual_weight_kg':   None,
        'note':               '',
    },
    {
        'batch_id':           'BATCH-F-02',
        'sow_date':           '2025-03-01',
        'transplant_date':    '2025-03-08',
        'plant_date':         '2025-03-10',
        'harvest_date':       str(TARGET_DATES[0]),
        'grow_days':          17,
        'bed_type':           'fixed',
        'bed_id':             5,
        'tray_or_gutter':     40,
        'weight_per_plant_g': 100,
        'loss_rate':          None,
        'actual_yield':       None,
        'actual_weight_kg':   None,
        'note':               '',
    },
    {
        'batch_id':           'BATCH-F-03',
        'sow_date':           '2025-03-02',
        'transplant_date':    '2025-03-09',
        'plant_date':         '2025-03-11',
        'harvest_date':       str(TARGET_DATES[1]),
        'grow_days':          17,
        'bed_type':           'fixed',
        'bed_id':             8,
        'tray_or_gutter':     32,
        'weight_per_plant_g': 95,
        'loss_rate':          None,
        'actual_yield':       None,
        'actual_weight_kg':   None,
        'note':               '',
    },
    {
        'batch_id':           'BATCH-F-04',
        'sow_date':           '2025-03-02',
        'transplant_date':    '2025-03-09',
        'plant_date':         '2025-03-11',
        'harvest_date':       str(TARGET_DATES[1]),
        'grow_days':          17,
        'bed_type':           'fixed',
        'bed_id':             15,
        'tray_or_gutter':     32,
        'weight_per_plant_g': None,
        'loss_rate':          None,
        'actual_yield':       None,
        'actual_weight_kg':   None,
        'note':               '주당 무게 미입력 → 기본값 적용',
    },
    {
        'batch_id':           'BATCH-MGS-A1',
        'sow_date':           '2025-03-02',
        'transplant_date':    '2025-03-09',
        'plant_date':         '2025-03-11',
        'harvest_date':       str(TARGET_DATES[1]),
        'grow_days':          17,
        'bed_type':           'mgs',
        'bed_id':             'MGS-A',
        'tray_or_gutter':     None,
        'weight_per_plant_g': 100,
        'loss_rate':          None,
        'actual_yield':       None,
        'actual_weight_kg':   None,
        'note':               '거터 수 미확정',
    },
]

DELETE_IDS = []

# ── 실행 로직 ─────────────────────────────────────────────────
db = pd.read_csv(DB_PATH, encoding='utf-8-sig') if os.path.exists(DB_PATH) else pd.DataFrame(columns=DB_COLS)

if MODE == 'view':
    print(f'📋 현재 DB: {len(db)}건')
    display(db)

elif MODE == 'add':
    new_df = pd.DataFrame(NEW_BATCHES)
    for c in DB_COLS:
        if c not in new_df.columns: new_df[c] = None
    new_df = new_df[DB_COLS]
    db = db[~db['batch_id'].isin(new_df['batch_id'].tolist())]
    db = pd.concat([db, new_df], ignore_index=True)
    db.to_csv(DB_PATH, index=False, encoding='utf-8-sig')
    print(f'✅ {len(new_df)}건 추가/업데이트  (DB 총 {len(db)}건)')
    print(f'   저장 위치: {DB_PATH}')
    display(db.tail(len(new_df)))

elif MODE == 'delete':
    before = len(db)
    db = db[~db['batch_id'].isin(DELETE_IDS)]
    db.to_csv(DB_PATH, index=False, encoding='utf-8-sig')
    print(f'🗑️  {before - len(db)}건 삭제  (DB 총 {len(db)}건)')

elif MODE == 'clear':
    if input('⚠️  DB 전체 삭제. YES 입력: ').strip() == 'YES':
        pd.DataFrame(columns=DB_COLS).to_csv(DB_PATH, index=False, encoding='utf-8-sig')
        print('🗑️  초기화 완료')
    else:
        print('취소')

In [ ]:
# ============================================================
# 셀 5: 📊 3일 대시보드 (D+0 / D+3 / D+4)
# ─ 배치 상세: 파종일 · 총 재배일수 포함
# ─ actual_weight_kg 입력 시 예측 vs 실적 비교 자동 표시
# ============================================================

# ── 공통 헬퍼 ────────────────────────────────────────────────
def load_db():
    """DB 로드 + 날짜/수치 파싱 + 기본값 채우기"""
    d = pd.read_csv(DB_PATH, encoding='utf-8-sig')
    for col in ['sow_date','transplant_date','plant_date','harvest_date']:
        d[col] = pd.to_datetime(d[col], errors='coerce')
    d['loss_rate']          = pd.to_numeric(d['loss_rate'],          errors='coerce').fillna(LOSS_RATE)
    d['tray_or_gutter']     = pd.to_numeric(d['tray_or_gutter'],     errors='coerce')
    d['weight_per_plant_g'] = pd.to_numeric(d['weight_per_plant_g'], errors='coerce').fillna(DEFAULT_WEIGHT_G)
    d['actual_yield']       = pd.to_numeric(d['actual_yield'],       errors='coerce')
    d['actual_weight_kg']   = pd.to_numeric(d['actual_weight_kg'],   errors='coerce')
    # MGS 주당 무게 별도 설정 적용
    d.loc[d['bed_type'] == 'mgs', 'weight_per_plant_g'] = d.loc[d['bed_type'] == 'mgs', 'weight_per_plant_g'].fillna(MGS_DEFAULT_WEIGHT_G)
    d.loc[(d['bed_type'] == 'mgs') & (d['weight_per_plant_g'] == DEFAULT_WEIGHT_G), 'weight_per_plant_g'] = MGS_DEFAULT_WEIGHT_G
    # 날짜 역전 제거
    bad = d[d['harvest_date'] < d['plant_date']]
    if not bad.empty:
        print(f'[⚠️] 날짜 역전 {len(bad)}건 제외')
        d = d[d['harvest_date'] >= d['plant_date']]
    return d

def calc_predictions(d):
    """예측 주수·kg 계산 컬럼 추가"""
    def _row(r):
        tg = r['tray_or_gutter']
        if pd.isna(tg): return pd.Series({'predicted_plants': None, 'predicted_kg': None})
        ppu = PLANTS_PER_TRAY if r['bed_type'] == 'fixed' else PLANTS_PER_GUTTER
        p   = round(float(tg) * ppu * (1 - r['loss_rate']))
        k   = round(p * float(r['weight_per_plant_g']) / 1000, 2)
        return pd.Series({'predicted_plants': p, 'predicted_kg': k})
    d[['predicted_plants','predicted_kg']] = d.apply(_row, axis=1)
    # 총 재배일수 = 파종일 ~ 수확일
    d['total_days'] = (d['harvest_date'] - d['sow_date']).dt.days
    d['status'] = d['predicted_plants'].apply(
        lambda v: '규칙 기반' if pd.notna(v) else 'N/A')
    return d

# ── 데이터 준비 ───────────────────────────────────────────────
db_all = load_db()
if db_all.empty:
    print('⚠️  DB가 비어 있습니다. 셀 4에서 배치를 추가하세요.')
else:
    db_all = calc_predictions(db_all)
    target = db_all[db_all['harvest_date'].dt.date.isin(TARGET_DATES)].copy()

    # ── 날짜별 합계 ───────────────────────────────────────────
    def day_sum(d, target_date):
        sub = d[d['harvest_date'].dt.date == target_date]
        pp = sub['predicted_plants'].sum(skipna=True)
        pk = sub['predicted_kg'].sum(skipna=True)
        ap = sub['actual_yield'].sum(skipna=True)
        ak = sub['actual_weight_kg'].sum(skipna=True)
        has_actual = sub['actual_weight_kg'].notna().any()
        return (int(pp) if pp else 0, round(float(pk),1) if pk else 0.0,
                int(ap) if ap else None, round(float(ak),1) if ak and has_actual else None,
                has_actual)

    d0_pp, d0_pk, d0_ap, d0_ak, d0_ha = day_sum(db_all, DASH_DATES[0])
    d3_pp, d3_pk, d3_ap, d3_ak, d3_ha = day_sum(db_all, DASH_DATES[1])
    d4_pp, d4_pk, d4_ap, d4_ak, d4_ha = day_sum(db_all, DASH_DATES[2])
    total_pp = d3_pp + d4_pp
    total_pk = round(d3_pk + d4_pk, 1)
    valid    = target[target['predicted_plants'].notna()]
    fixed_pp = int(valid[valid['bed_type']=='fixed']['predicted_plants'].sum())
    fixed_pk = round(float(valid[valid['bed_type']=='fixed']['predicted_kg'].sum()),1)
    mgs_pp   = int(valid[valid['bed_type']=='mgs']['predicted_plants'].sum())
    mgs_pk   = round(float(valid[valid['bed_type']=='mgs']['predicted_kg'].sum()),1)
    has_na   = target['predicted_plants'].isna().any()
    any_actual = target['actual_weight_kg'].notna().any()

    # ── 날짜 카드 HTML 생성 ────────────────────────────────────
    WEEKDAYS_KR = ['월','화','수','목','금','토','일']
    def wd(d): return WEEKDAYS_KR[d.weekday()]

    def day_card(label, color, dt, pp, pk, ap, ak, has_actual):
        border = f'border:2px solid {color}' if color != '#888' else 'border:0.5px solid #ccc'
        tag_c  = color
        date_s = f'{dt.month}월 {dt.day}일 ({wd(dt)})'
        pred_p = f'{pp:,}주' if pp else ('수확 없음' if not has_actual else '—')
        pred_k = f'{pk:.1f} kg' if pk else '—'
        act_p  = f'{ap:,}주' if ap is not None else '—'
        act_k  = f'{ak:.1f} kg' if ak is not None else '—'
        diff_k = ''
        diff_style = ''
        if pk and ak is not None:
            d_val  = round(ak - pk, 1)
            sign   = '+' if d_val >= 0 else ''
            diff_k = f'{sign}{d_val} kg'
            diff_style = 'color:#27500A' if d_val >= 0 else 'color:#A32D2D'
        actual_block = ''
        if has_actual:
            actual_block = f'''
              <div style="border-top:0.5px solid #e0e0e0;margin-top:10px;padding-top:10px">
                <div style="font-size:10px;font-weight:600;color:#888;letter-spacing:.6px;margin-bottom:6px">실적</div>
                <div style="display:flex;justify-content:space-between;align-items:baseline;margin-bottom:4px">
                  <span style="font-size:11px;color:#888">실제 주수</span>
                  <span style="font-size:15px;font-weight:500">{act_p}</span>
                </div>
                <div style="display:flex;justify-content:space-between;align-items:baseline;margin-bottom:4px">
                  <span style="font-size:11px;color:#888">실제 무게</span>
                  <span style="font-size:15px;font-weight:500;color:#185FA5">{act_k}</span>
                </div>
                {f'<div style="text-align:right;font-size:12px;font-weight:500;{diff_style}">오차 {diff_k}</div>' if diff_k else ''}
              </div>'''
        return f'''
        <div style="background:#fff;{border};border-radius:12px;padding:14px 16px">
          <div style="font-size:10px;font-weight:600;letter-spacing:.8px;color:{tag_c};margin-bottom:6px">{label}</div>
          <div style="font-size:14px;font-weight:500;margin-bottom:12px">{date_s}</div>
          <div style="font-size:10px;font-weight:600;color:#888;letter-spacing:.6px;margin-bottom:6px">예측</div>
          <div style="display:flex;justify-content:space-between;align-items:baseline;margin-bottom:4px">
            <span style="font-size:11px;color:#888">주수</span>
            <span style="font-size:{'17' if pp else '13'}px;font-weight:500;color:{'#1a1a2e' if pp else '#aaa'}">{pred_p}</span>
          </div>
          <div style="display:flex;justify-content:space-between;align-items:baseline">
            <span style="font-size:11px;color:#888">무게</span>
            <span style="font-size:17px;font-weight:500;color:{color if pk else '#aaa'}">{pred_k}</span>
          </div>
          {actual_block}
        </div>'''

    card0 = day_card('D+0 · 오늘','#888',DASH_DATES[0],d0_pp,d0_pk,d0_ap,d0_ak,d0_ha)
    card3 = day_card('D+3 · 수확 예정','#3B6D11',DASH_DATES[1],d3_pp,d3_pk,d3_ap,d3_ak,d3_ha)
    card4 = day_card('D+4 · 수확 예정','#185FA5',DASH_DATES[2],d4_pp,d4_pk,d4_ap,d4_ak,d4_ha)

    # ── 배치 상세 테이블 ─────────────────────────────────────────
    def fmt_date(v):
        try: return pd.Timestamp(v).strftime('%m-%d')
        except: return '—'

    detail_rows = ''
    for _, row in target.sort_values('harvest_date').iterrows():
        bt_lbl   = '고정' if row['bed_type']=='fixed' else 'MGS'
        bt_bg    = '#EAF3DE' if row['bed_type']=='fixed' else '#E6F1FB'
        bt_fg    = '#27500A' if row['bed_type']=='fixed' else '#0C447C'
        hdate    = fmt_date(row['harvest_date'])
        sdate    = fmt_date(row['sow_date'])
        tdays    = int(row['total_days']) if pd.notna(row['total_days']) else '—'
        tg       = int(row['tray_or_gutter']) if pd.notna(row['tray_or_gutter']) else '—'
        wpg      = f"{int(row['weight_per_plant_g'])}g" if pd.notna(row['weight_per_plant_g']) else '—'
        loss_pct = f"{row['loss_rate']*100:.0f}%"
        pp       = f"{int(row['predicted_plants']):,}" if pd.notna(row['predicted_plants']) else '<span style="color:#aaa">N/A</span>'
        pk       = f"{row['predicted_kg']:.1f}" if pd.notna(row['predicted_kg']) else '<span style="color:#aaa">N/A</span>'
        ap       = f"{int(row['actual_yield']):,}" if pd.notna(row['actual_yield']) else ''
        ak       = f"{row['actual_weight_kg']:.1f}" if pd.notna(row['actual_weight_kg']) else ''
        awpg = round(float(row['actual_weight_kg']) * 1000 / float(row['actual_yield']), 1) if (pd.notna(row['actual_yield']) and pd.notna(row['actual_weight_kg']) and float(row['actual_yield']) > 0) else None
        # 실적 오차
        diff_html = ''
        if pd.notna(row['actual_weight_kg']) and pd.notna(row['predicted_kg']):
            diff = round(float(row['actual_weight_kg']) - float(row['predicted_kg']), 1)
            sign = '+' if diff >= 0 else ''
            clr  = '#27500A' if diff >= 0 else '#A32D2D'
            diff_html = f'<br><span style="font-size:10px;color:{clr}">{sign}{diff} kg</span>'
        note_txt = str(row.get('note','')) if pd.notna(row.get('note','')) else row.get('status','')
        detail_rows += f'''
        <tr>
          <td style="font-family:monospace;font-size:11px">{row["batch_id"]}</td>
          <td><span style="background:{bt_bg};color:{bt_fg};padding:2px 7px;border-radius:10px;font-size:11px;font-weight:600">{bt_lbl}</span></td>
          <td style="text-align:center">{row["bed_id"]}</td>
          <td style="text-align:center">{sdate}</td>
          <td style="text-align:center">{hdate}</td>
          <td style="text-align:center">{tdays}일</td>
          <td style="text-align:center">{tg}</td>
          <td style="text-align:center">{wpg}</td>
          <td style="text-align:center">{loss_pct}</td>
          <td style="text-align:right;font-weight:500">{pp}주</td>
          <td style="text-align:right;font-weight:500;color:#27500A">{pk} kg{diff_html}</td>
          <td style="text-align:right;color:#185FA5">{ap}{('<br>' if ap and ak else '')}{ak + ' kg' if ak else ''}</td>
          <td style="text-align:right;color:#185FA5">{f'awpg:.1f} g' if awpg is not None else '—'}</td>
          <td style="font-size:11px;color:#999">{note_txt}</td>
        </tr>'''

    na_warn = ''
    if has_na:
        na_warn = '<p style="color:#854F0B;background:#FAEEDA;padding:8px 12px;border-radius:8px;font-size:12px;margin-top:10px">⚠️ 거터 수 미입력 배치 포함 — 셀 4에서 tray_or_gutter(거터 수) 입력 후 재실행</p>'

    actual_note = ''
    if any_actual:
        actual_note = '<p style="font-size:12px;color:#0C447C;background:#E6F1FB;padding:8px 12px;border-radius:8px;margin-top:10px">📥 실적 데이터 포함 — 카드 하단 및 무게 컬럼에 오차 표시됨</p>'

    html = f'''
    <div style="font-family:-apple-system,sans-serif;max-width:960px">
      <div style="font-size:11px;color:#999;margin-bottom:16px">
        예측 기준일: {PREDICTION_DATE} &nbsp;·&nbsp; 로스율 {LOSS_RATE_PCT}% &nbsp;·&nbsp; 주당 기본 무게 {DEFAULT_WEIGHT_G}g
      </div>

      <div style="display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:10px;margin-bottom:14px">
        {card0}{card3}{card4}
      </div>

      <div style="background:#27500A;border-radius:12px;padding:14px 20px;
                  display:flex;justify-content:space-around;margin-bottom:14px">
        <div style="text-align:center">
          <div style="font-size:10px;color:#C0DD97;margin-bottom:4px">이번 주 예측 주수 (D+3~4)</div>
          <div style="font-size:20px;font-weight:500;color:#EAF3DE">{total_pp:,}주</div>
        </div>
        <div style="text-align:center">
          <div style="font-size:10px;color:#C0DD97;margin-bottom:4px">이번 주 예측 무게 (D+3~4)</div>
          <div style="font-size:20px;font-weight:500;color:#EAF3DE">{total_pk:.1f} kg</div>
        </div>
        <div style="text-align:center">
          <div style="font-size:10px;color:#C0DD97;margin-bottom:4px">수확률</div>
          <div style="font-size:20px;font-weight:500;color:#EAF3DE">{100-LOSS_RATE_PCT}%</div>
          <div style="font-size:10px;color:#97C459">로스율 {LOSS_RATE_PCT}%</div>
        </div>
      </div>

      <div style="display:grid;grid-template-columns:1fr 1fr;gap:10px;margin-bottom:14px">
        <div style="border:0.5px solid #e0e0e0;border-radius:8px;padding:12px 16px">
          <div style="font-size:11px;font-weight:600;color:#555;margin-bottom:8px">고정 재배대</div>
          <div style="display:flex;gap:20px">
            <div style="text-align:center"><div style="font-size:16px;font-weight:500">{fixed_pp:,}주</div><div style="font-size:10px;color:#999">예측 주수</div></div>
            <div style="text-align:center"><div style="font-size:16px;font-weight:500;color:#27500A">{fixed_pk:.1f} kg</div><div style="font-size:10px;color:#999">예측 무게</div></div>
          </div>
        </div>
        <div style="border:0.5px solid #e0e0e0;border-radius:8px;padding:12px 16px">
          <div style="font-size:11px;font-weight:600;color:#555;margin-bottom:8px">MGS (NFT)</div>
          <div style="display:flex;gap:20px">
            <div style="text-align:center"><div style="font-size:16px;font-weight:500;{'color:#aaa' if not mgs_pp else ''}">{f'{mgs_pp:,}주' if mgs_pp else 'N/A'}</div><div style="font-size:10px;color:#999">예측 주수</div></div>
            <div style="text-align:center"><div style="font-size:16px;font-weight:500;{'color:#aaa' if not mgs_pk else 'color:#185FA5'}">{f'{mgs_pk:.1f} kg' if mgs_pk else 'N/A'}</div><div style="font-size:10px;color:#999">예측 무게</div></div>
          </div>
        </div>
      </div>

      <div style="font-size:13px;font-weight:500;color:#444;margin-bottom:8px">배치별 상세 (D+3~4)</div>
      <div style="overflow-x:auto">
      <table style="width:100%;border-collapse:collapse;font-size:12px;white-space:nowrap">
        <thead>
          <tr style="background:#f7f7f7">
            <th style="padding:7px 8px;text-align:left;border-bottom:1.5px solid #ddd;font-size:11px;color:#555;font-weight:600">배치 ID</th>
            <th style="padding:7px 8px;text-align:left;border-bottom:1.5px solid #ddd;font-size:11px;color:#555;font-weight:600">방식</th>
            <th style="padding:7px 8px;text-align:center;border-bottom:1.5px solid #ddd;font-size:11px;color:#555;font-weight:600">재배대</th>
            <th style="padding:7px 8px;text-align:center;border-bottom:1.5px solid #ddd;font-size:11px;color:#555;font-weight:600">파종일</th>
            <th style="padding:7px 8px;text-align:center;border-bottom:1.5px solid #ddd;font-size:11px;color:#555;font-weight:600">수확예정일</th>
            <th style="padding:7px 8px;text-align:center;border-bottom:1.5px solid #ddd;font-size:11px;color:#555;font-weight:600">총 재배일</th>
            <th style="padding:7px 8px;text-align:center;border-bottom:1.5px solid #ddd;font-size:11px;color:#555;font-weight:600">판/거터</th>
            <th style="padding:7px 8px;text-align:center;border-bottom:1.5px solid #ddd;font-size:11px;color:#555;font-weight:600">주당 무게</th>
            <th style="padding:7px 8px;text-align:center;border-bottom:1.5px solid #ddd;font-size:11px;color:#555;font-weight:600">로스율</th>
            <th style="padding:7px 8px;text-align:right;border-bottom:1.5px solid #ddd;font-size:11px;color:#555;font-weight:600">예측 주수</th>
            <th style="padding:7px 8px;text-align:right;border-bottom:1.5px solid #ddd;font-size:11px;color:#27500A;font-weight:600">예측 무게</th>
            <th style="padding:7px 8px;text-align:right;border-bottom:1.5px solid #ddd;font-size:11px;color:#185FA5;font-weight:600">실적</th>
                        <th style="padding:7px 8px;text-align:right;border-bottom:1.5px solid #ddd;font-size:11px;color:#185FA5;font-weight:600">실제 주당무게</th>
            <th style="padding:7px 8px;text-align:left;border-bottom:1.5px solid #ddd;font-size:11px;color:#555;font-weight:600">비고</th>
          </tr>
        </thead>
        <tbody>{detail_rows}</tbody>
      </table>
      </div>
      {na_warn}{actual_note}
    </div>
    '''
    display(HTML(html))

    # 드라이브 저장
    result_path = os.path.join(RESULT_DIR, f'prediction_{PREDICTION_DATE}.csv')
    save_cols = ['batch_id','bed_type','bed_id','sow_date','harvest_date','total_days',
                 'tray_or_gutter','weight_per_plant_g','loss_rate',
                 'predicted_plants','predicted_kg','actual_yield','actual_weight_kg','status','note']
    target[[c for c in save_cols if c in target.columns]].to_csv(
        result_path, index=False, encoding='utf-8-sig')
    print(f'\n💾 저장: {result_path}')

In [ ]:
# ============================================================
# 셀 6: 📅 달별 전체 예측 뷰
# ─ DB에 있는 모든 월을 수확일 기준으로 집계
# ─ 월별 요약 + 배치 상세 (파종일·총 재배일수 포함)
# ============================================================

try:
    db_all
except NameError:
    db_all = calc_predictions(load_db())

if db_all.empty:
    print('⚠️  DB가 비어 있습니다.')
else:
    # 수확 예정일이 있는 배치만
    df_m = db_all[db_all['harvest_date'].notna()].copy()
    df_m['ym'] = df_m['harvest_date'].dt.to_period('M')
    months = sorted(df_m['ym'].unique())

    month_blocks = ''
    for ym in months:
        sub = df_m[df_m['ym'] == ym].sort_values('harvest_date')
        m_pp   = int(sub['predicted_plants'].sum(skipna=True))
        m_pk   = round(float(sub['predicted_kg'].sum(skipna=True)), 1)
        m_ap   = sub['actual_yield'].sum(skipna=True)
        m_ak   = sub['actual_weight_kg'].sum(skipna=True)
        has_ak = sub['actual_weight_kg'].notna().any()
        m_ap   = int(m_ap) if m_ap else None
        m_ak   = round(float(m_ak), 1) if has_ak else None
        n_na   = sub['predicted_plants'].isna().sum()

        # 월 요약 바
        actual_summary = ''
        if has_ak:
            diff = round(float(m_ak) - float(m_pk), 1) if m_ak and m_pk else None
            sign = '+' if diff and diff >= 0 else ''
            diff_c = '#C0DD97' if diff and diff >= 0 else '#F09595'
            actual_summary = f'''
            <div style="text-align:center">
              <div style="font-size:10px;color:#C0DD97;margin-bottom:4px">실제 무게</div>
              <div style="font-size:18px;font-weight:500;color:#EAF3DE">{m_ak:.1f} kg</div>
            </div>
            <div style="text-align:center">
              <div style="font-size:10px;color:#C0DD97;margin-bottom:4px">오차</div>
              <div style="font-size:18px;font-weight:500;color:{diff_c}">{sign}{diff} kg</div>
            </div>'''

        # 배치 상세 행
        batch_rows = ''
        for _, row in sub.iterrows():
            bt_lbl = '고정' if row['bed_type']=='fixed' else 'MGS'
            bt_bg  = '#EAF3DE' if row['bed_type']=='fixed' else '#E6F1FB'
            bt_fg  = '#27500A' if row['bed_type']=='fixed' else '#0C447C'
            hdate  = row['harvest_date'].strftime('%m-%d') if pd.notna(row['harvest_date']) else '—'
            sdate  = row['sow_date'].strftime('%m-%d') if pd.notna(row['sow_date']) else '—'
            tdays  = f"{int(row['total_days'])}일" if pd.notna(row['total_days']) else '—'
            tg     = int(row['tray_or_gutter']) if pd.notna(row['tray_or_gutter']) else '—'
            wpg    = f"{int(row['weight_per_plant_g'])}g" if pd.notna(row['weight_per_plant_g']) else '—'
            loss_p = f"{row['loss_rate']*100:.0f}%"
            pp_s   = f"{int(row['predicted_plants']):,}주" if pd.notna(row['predicted_plants']) else '<span style="color:#aaa">N/A</span>'
            pk_s   = f"{row['predicted_kg']:.1f} kg" if pd.notna(row['predicted_kg']) else '<span style="color:#aaa">N/A</span>'
            ap_s   = f"{int(row['actual_yield']):,}주" if pd.notna(row['actual_yield']) else '—'
            ak_s   = f"{row['actual_weight_kg']:.1f} kg" if pd.notna(row['actual_weight_kg']) else '—'
            awpg_m = round(float(row['actual_weight_kg']) * 1000 / float(row['actual_yield']), 1) if (pd.notna(row['actual_yield']) and pd.notna(row['actual_weight_kg']) and float(row['actual_yield']) > 0) else None
            # 실적 오차
            ak_cell = ak_s
            if pd.notna(row['actual_weight_kg']) and pd.notna(row['predicted_kg']):
                d_val = round(float(row['actual_weight_kg']) - float(row['predicted_kg']), 1)
                sign  = '+' if d_val >= 0 else ''
                clr   = '#27500A' if d_val >= 0 else '#A32D2D'
                ak_cell = f"{ak_s}<br><span style='font-size:10px;color:{clr}'>{sign}{d_val} kg</span>"
            note_txt = str(row.get('note','')) if pd.notna(row.get('note','')) else ''
            batch_rows += f'''
            <tr style="border-bottom:0.5px solid #eee">
              <td style="padding:7px 8px;font-family:monospace;font-size:11px">{row["batch_id"]}</td>
              <td style="padding:7px 8px"><span style="background:{bt_bg};color:{bt_fg};padding:2px 7px;border-radius:10px;font-size:11px;font-weight:600">{bt_lbl}</span></td>
              <td style="padding:7px 8px;text-align:center">{row["bed_id"]}</td>
              <td style="padding:7px 8px;text-align:center">{sdate}</td>
              <td style="padding:7px 8px;text-align:center">{hdate}</td>
              <td style="padding:7px 8px;text-align:center">{tdays}</td>
              <td style="padding:7px 8px;text-align:center">{tg}</td>
              <td style="padding:7px 8px;text-align:center">{wpg}</td>
              <td style="padding:7px 8px;text-align:center">{loss_p}</td>
              <td style="padding:7px 8px;text-align:right;font-weight:500">{pp_s}</td>
              <td style="padding:7px 8px;text-align:right;font-weight:500;color:#27500A">{pk_s}</td>
              <td style="padding:7px 8px;text-align:right;color:#185FA5">{ap_s}</td>
              <td style="padding:7px 8px;text-align:right">{ak_cell}</td>
              <td style="padding:7px 8px;font-size:11px;color:#999">{note_txt}</td>
            </tr>'''

        na_note = f'<span style="font-size:11px;color:#854F0B"> · N/A {n_na}건 포함</span>' if n_na else ''

        month_blocks += f'''
        <div style="margin-bottom:24px">
          <div style="background:#27500A;border-radius:12px;padding:12px 20px;
                      display:flex;align-items:center;justify-content:space-between;margin-bottom:10px">
            <div style="font-size:16px;font-weight:500;color:#EAF3DE">{ym.year}년 {ym.month}월{na_note}</div>
            <div style="display:flex;gap:24px">
              <div style="text-align:center">
                <div style="font-size:10px;color:#C0DD97;margin-bottom:2px">예측 주수</div>
                <div style="font-size:18px;font-weight:500;color:#EAF3DE">{m_pp:,}주</div>
              </div>
              <div style="text-align:center">
                <div style="font-size:10px;color:#C0DD97;margin-bottom:2px">예측 무게</div>
                <div style="font-size:18px;font-weight:500;color:#EAF3DE">{m_pk:.1f} kg</div>
              </div>
              {actual_summary}
              <div style="text-align:center">
                <div style="font-size:10px;color:#C0DD97;margin-bottom:2px">배치 수</div>
                <div style="font-size:18px;font-weight:500;color:#EAF3DE">{len(sub)}건</div>
              </div>
            </div>
          </div>
          <div style="overflow-x:auto">
          <table style="width:100%;border-collapse:collapse;font-size:12px;white-space:nowrap">
            <thead>
              <tr style="background:#f7f7f7">
                <th style="padding:6px 8px;text-align:left;border-bottom:1.5px solid #ddd;font-size:11px;color:#555">배치 ID</th>
                <th style="padding:6px 8px;text-align:left;border-bottom:1.5px solid #ddd;font-size:11px;color:#555">방식</th>
                <th style="padding:6px 8px;text-align:center;border-bottom:1.5px solid #ddd;font-size:11px;color:#555">재배대</th>
                <th style="padding:6px 8px;text-align:center;border-bottom:1.5px solid #ddd;font-size:11px;color:#555">파종일</th>
                <th style="padding:6px 8px;text-align:center;border-bottom:1.5px solid #ddd;font-size:11px;color:#555">수확예정일</th>
                <th style="padding:6px 8px;text-align:center;border-bottom:1.5px solid #ddd;font-size:11px;color:#555">총 재배일</th>
                <th style="padding:6px 8px;text-align:center;border-bottom:1.5px solid #ddd;font-size:11px;color:#555">판/거터</th>
                <th style="padding:6px 8px;text-align:center;border-bottom:1.5px solid #ddd;font-size:11px;color:#555">주당 무게</th>
                <th style="padding:6px 8px;text-align:center;border-bottom:1.5px solid #ddd;font-size:11px;color:#555">로스율</th>
                <th style="padding:6px 8px;text-align:right;border-bottom:1.5px solid #ddd;font-size:11px;color:#555">예측 주수</th>
                <th style="padding:6px 8px;text-align:right;border-bottom:1.5px solid #ddd;font-size:11px;color:#27500A">예측 무게</th>
                <th style="padding:6px 8px;text-align:right;border-bottom:1.5px solid #ddd;font-size:11px;color:#185FA5">실제 주수</th>
                <th style="padding:6px 8px;text-align:right;border-bottom:1.5px solid #ddd;font-size:11px;color:#185FA5">실제 무게</th>
                                <th style="padding:6px 8px;text-align:right;border-bottom:1.5px solid #ddd;font-size:11px;color:#185FA5;font-weight:600">실제 주당무게</th>
                <th style="padding:6px 8px;text-align:left;border-bottom:1.5px solid #ddd;font-size:11px;color:#555">비고</th>
              </tr>
            </thead>
            <tbody>{batch_rows}</tbody>
          </table>
          </div>
        </div>'''

    total_all_pp = int(df_m['predicted_plants'].sum(skipna=True))
    total_all_pk = round(float(df_m['predicted_kg'].sum(skipna=True)), 1)

    html_monthly = f'''
    <div style="font-family:-apple-system,sans-serif;max-width:980px">
      <div style="font-size:18px;font-weight:500;margin-bottom:4px">달별 전체 예측 현황</div>
      <div style="font-size:11px;color:#999;margin-bottom:6px">
        DB 전체 {len(df_m)}건 · 로스율 {LOSS_RATE_PCT}% · 주당 기본 무게 {DEFAULT_WEIGHT_G}g
      </div>
      <div style="display:inline-flex;gap:20px;background:#f4f4f4;border-radius:8px;padding:10px 16px;margin-bottom:20px">
        <div><span style="font-size:11px;color:#888">전체 예측 주수 &nbsp;</span><strong>{total_all_pp:,}주</strong></div>
        <div><span style="font-size:11px;color:#888">전체 예측 무게 &nbsp;</span><strong style="color:#27500A">{total_all_pk:.1f} kg</strong></div>
        <div><span style="font-size:11px;color:#888">기간 &nbsp;</span><strong>{months[0]} ~ {months[-1]}</strong></div>
      </div>
      {month_blocks}
    </div>
    '''
    display(HTML(html_monthly))

In [ ]:
# ============================================================
# 셀 7: ✍️ 실적 입력 (수확 완료 후)
# ─ actual_weight_kg 를 입력하면 셀 5/6 대시보드에 자동 반영
# ─ 셀 5를 다시 실행하면 예측 vs 실적 비교 즉시 표시
# ============================================================

# batch_id: (실제 주수, 실제 무게 kg)
ACTUAL_RESULTS = {
    # 예시:
    # 'BATCH-F-01': (498, 48.3),
    # 'BATCH-F-02': (510, 51.2),
}

if not ACTUAL_RESULTS:
    print('ℹ️  ACTUAL_RESULTS에 실적 값을 입력한 뒤 실행하세요.')
    print('   형식: \'배치ID\': (주수, 무게kg)')
else:
    db_w = pd.read_csv(DB_PATH, encoding='utf-8-sig')
    updated = 0
    for bid, (plants, kg) in ACTUAL_RESULTS.items():
        mask = db_w['batch_id'] == bid
        if mask.any():
            db_w.loc[mask, 'actual_yield']     = plants
            db_w.loc[mask, 'actual_weight_kg'] = kg
            updated += 1
            print(f'  ✅ {bid} → 실제 {plants:,}주 / {kg:.1f} kg 저장')
        else:
            print(f'  ⚠️  {bid} → DB에 없음')
    db_w.to_csv(DB_PATH, index=False, encoding='utf-8-sig')
    print(f'\n💾 실적 {updated}건 저장 완료 → {DB_PATH}')
    print('\n👉 셀 5를 다시 실행하면 대시보드에 실적이 반영됩니다.')

In [ ]:
# ============================================================
# 셀 8: 📋 노션 마크다운 출력 (D+3~4 기준)
# ============================================================
try:
    md = [
        f'## 수확량 예측 — {PREDICTION_DATE} 기준',
        '',
        '| 수확 예정일 | 방식 | 재배대 | 파종일 | 총 재배일 | 예측 주수 | 예측 무게(kg) | 주당 무게(g) | 로스율 | 실적(kg) | 비고 |',
        '|---|---|---|---|---|---|---|---|---|---|---|',
    ]
    for _, row in target.sort_values('harvest_date').iterrows():
        hd    = str(row['harvest_date'].date())
        sd    = str(row['sow_date'].date()) if pd.notna(row['sow_date']) else '—'
        td    = f"{int(row['total_days'])}일" if pd.notna(row['total_days']) else '—'
        btype = '고정 재배대' if row['bed_type'] == 'fixed' else 'MGS'
        pp    = f"{int(row['predicted_plants']):,}주" if pd.notna(row['predicted_plants']) else 'N/A'
        pk    = f"{row['predicted_kg']:.1f}" if pd.notna(row['predicted_kg']) else 'N/A'
        wpg   = f"{int(row['weight_per_plant_g'])}g" if pd.notna(row['weight_per_plant_g']) else '—'
        loss  = f"{row['loss_rate']*100:.0f}%"
        ak    = f"{row['actual_weight_kg']:.1f}" if pd.notna(row['actual_weight_kg']) else '—'
        note  = str(row.get('note','')) if pd.notna(row.get('note','')) else ''
        md.append(f'| {hd} | {btype} | {row["bed_id"]} | {sd} | {td} | {pp} | {pk} | {wpg} | {loss} | {ak} | {note} |')
    md.append(f'| **합계** | | | | | **{total_pp:,}주** | **{total_pk:.1f}** | | | | |')
    print('📋 노션에 붙여넣으세요:\n')
    print('\n'.join(md))
except NameError:
    print('⚠️  셀 5를 먼저 실행해주세요.')

In [ ]:
# ============================================================
# [참고] 고정 재배대 전체 용량표
# ============================================================
cap_rows = []
for bid, trays in FIXED_BED_CONFIG.items():
    max_p  = trays * PLANTS_PER_TRAY
    pred_p = round(max_p * YIELD_RATE)
    pred_k = round(pred_p * DEFAULT_WEIGHT_G / 1000, 1)
    cap_rows.append({
        '재배대': f'{bid}번', '판 수': trays,
        '최대 식재': max_p,
        f'예측 주수({100-LOSS_RATE_PCT}%)': pred_p,
        f'예측 kg({DEFAULT_WEIGHT_G}g/주)': pred_k,
    })
cap_df  = pd.DataFrame(cap_rows)
tot_row = {
    '재배대': '합계', '판 수': cap_df['판 수'].sum(),
    '최대 식재': cap_df['최대 식재'].sum(),
    f'예측 주수({100-LOSS_RATE_PCT}%)': cap_df[f'예측 주수({100-LOSS_RATE_PCT}%)'].sum(),
    f'예측 kg({DEFAULT_WEIGHT_G}g/주)': round(cap_df[f'예측 kg({DEFAULT_WEIGHT_G}g/주)'].sum(), 1),
}
cap_df = pd.concat([cap_df, pd.DataFrame([tot_row])], ignore_index=True)
print(f'고정 재배대 용량표  (1판={PLANTS_PER_TRAY}주, 기본 {DEFAULT_WEIGHT_G}g/주, 수확률 {100-LOSS_RATE_PCT}%)')
display(cap_df)